# Healthcare Claims - Silver to Gold Transformations

This notebook reads cleansed Silver tables and produces aggregated Gold analytics tables, followed by pipeline validation.

**Dependency**: Runs the setup notebook first via `%run`.

In [0]:
%run ./healthcare_claims_setup

## Gold Layer - Aggregated Analytics

Business-ready analytics tables:
1. **patient_summary**: Patient demographics, enrollment periods, claims activity
2. **claims_analytics**: Claims trends by date, location, procedure type
3. **cost_analytics**: Financial metrics - charges vs allowed amounts by procedure/provider

In [0]:
# Gold Layer: Patient Summary Analytics
# Combines enrollment demographics with claims activity

# Read silver tables
enrollment_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.enrollment")
medical_claim_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")

# Aggregate claims per patient
claims_per_patient = (
    medical_claim_df
    .where(F.col("data_quality_flag") == "PASS")
    .groupBy("patient_id")
    .agg(
        F.count("*").alias("total_claims"),
        F.min("date_service").alias("first_claim_date"),
        F.max("date_service").alias("last_claim_date"),
        F.countDistinct("location_of_care").alias("distinct_locations"),
        F.countDistinct("pay_type").alias("distinct_pay_types")
    )
)

# Create patient summary by joining enrollment with claims
patient_summary_gold = (
    enrollment_df
    .where(F.col("data_quality_flag") == "PASS")
    .groupBy("patient_id")
    .agg(
        F.first("patient_gender").alias("gender"),
        F.first("patient_age").alias("age"),
        F.first("patient_zip3").alias("zip3"),
        F.first("patient_state").alias("state"),
        F.min("date_start").alias("enrollment_start"),
        F.max("date_end").alias("enrollment_end"),
        F.datediff(F.max("date_end"), F.min("date_start")).alias("enrollment_days"),
        F.countDistinct("benefit_type").alias("benefit_types_count"),
        F.collect_set("benefit_type").alias("benefit_types")
    )
    .join(claims_per_patient, "patient_id", "left")
    .withColumn("total_claims", F.coalesce(F.col("total_claims"), F.lit(0)))
    .withColumn("processed_timestamp", F.current_timestamp())
)

patient_summary_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.patient_summary"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.patient_summary")
print(f"  Total patients: {patient_summary_gold.count():,}")

In [0]:
# Gold Layer: Claims Analytics
# Aggregated claims by date, location, and procedure type

medical_claim_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.medical_claim")
procedure_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
diagnosis_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.diagnosis")

# Join claims with procedures and diagnosis
claims_enriched = (
    medical_claim_df
    .where(F.col("data_quality_flag") == "PASS")
    .join(
        procedure_df.where(F.col("data_quality_flag") == "PASS").drop("date_service", "date_service_end", "data_quality_flag", "processed_timestamp"),
        ["claim_id", "patient_id"],
        "left"
    )
    .join(
        diagnosis_df.where(F.col("data_quality_flag") == "PASS").drop("date_service", "date_service_end", "data_quality_flag", "processed_timestamp"),
        ["claim_id", "patient_id"],
        "left"
    )
)

# Aggregate by date and location
claims_analytics_gold = (
    claims_enriched
    .groupBy(
        F.year("date_service").alias("service_year"),
        F.month("date_service").alias("service_month"),
        "location_of_care",
        "pay_type"
    )
    .agg(
        F.count("claim_id").alias("total_claims"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.countDistinct("procedure_code").alias("unique_procedures"),
        F.countDistinct("diagnosis_code").alias("unique_diagnoses"),
        F.sum("line_charge").alias("total_charges"),
        F.sum("line_allowed").alias("total_allowed"),
        F.avg("line_charge").alias("avg_line_charge"),
        F.avg("line_allowed").alias("avg_line_allowed")
    )
    .withColumn(
        "charge_to_allowed_ratio",
        F.when(
            F.col("total_allowed") > 0,
            F.col("total_charges") / F.col("total_allowed")
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .orderBy("service_year", "service_month", "location_of_care")
)

claims_analytics_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.claims_analytics"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.claims_analytics")
print(f"  Analytics periods: {claims_analytics_gold.count():,}")

In [0]:
# Gold Layer: Cost Analytics
# Financial analysis by procedure and provider

procedure_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.procedure")
provider_df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.provider")

# Join procedures with providers for cost analysis
cost_analytics_gold = (
    procedure_df
    .where(F.col("data_quality_flag") == "PASS")
    .join(
        provider_df.where(F.col("data_quality_flag") == "PASS"),
        ["claim_id", "patient_id"],
        "left"
    )
    .groupBy(
        "procedure_code",
        "procedure_qual",
        "npi",
        "taxonomy_code"
    )
    .agg(
        F.count("*").alias("procedure_count"),
        F.countDistinct("patient_id").alias("unique_patients"),
        F.sum("line_charge").alias("total_charges"),
        F.sum("line_allowed").alias("total_allowed"),
        F.avg("line_charge").alias("avg_charge_per_procedure"),
        F.avg("line_allowed").alias("avg_allowed_per_procedure"),
        F.min("line_charge").alias("min_charge"),
        F.max("line_charge").alias("max_charge"),
        F.stddev("line_charge").alias("stddev_charge")
    )
    .withColumn(
        "charge_to_allowed_ratio",
        F.when(
            F.col("total_allowed") > 0,
            F.col("total_charges") / F.col("total_allowed")
        )
    )
    .withColumn(
        "discount_amount",
        F.col("total_charges") - F.col("total_allowed")
    )
    .withColumn(
        "discount_percentage",
        F.when(
            F.col("total_charges") > 0,
            ((F.col("total_charges") - F.col("total_allowed")) / F.col("total_charges")) * 100
        )
    )
    .withColumn("processed_timestamp", F.current_timestamp())
    .orderBy(F.desc("procedure_count"))
)

cost_analytics_gold.write.mode("overwrite").saveAsTable(
    f"{TARGET_CATALOG}.{GOLD_SCHEMA}.cost_analytics"
)

print(f"✓ Gold table created: {TARGET_CATALOG}.{GOLD_SCHEMA}.cost_analytics")
print(f"  Procedure/Provider combinations: {cost_analytics_gold.count():,}")

## Pipeline Summary & Validation

Validate the medallion architecture and check data quality metrics.

In [0]:
# Generate data quality summary across all Silver tables

silver_tables = ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]

print("\n" + "="*80)
print("SILVER LAYER - DATA QUALITY SUMMARY")
print("="*80 + "\n")

for table in silver_tables:
    df = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.{table}")
    total_records = df.count()
    passed_records = df.where(F.col("data_quality_flag") == "PASS").count()
    failed_records = df.where(F.col("data_quality_flag") == "FAIL").count()
    pass_rate = (passed_records / total_records * 100) if total_records > 0 else 0
    
    print(f"Table: {table}")
    print(f"  Total Records: {total_records:,}")
    print(f"  Passed: {passed_records:,} ({pass_rate:.2f}%)")
    print(f"  Failed: {failed_records:,}")
    print()

print("="*80)

In [0]:
# Summary of record counts across all layers

print("\n" + "="*80)
print("MEDALLION ARCHITECTURE - RECORD COUNTS")
print("="*80 + "\n")

# Bronze layer
print("🥉 BRONZE LAYER (Source)")
print(f"  Catalog.Schema: {BRONZE_CATALOG}.{BRONZE_SCHEMA}")
for table in ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]:
    count = spark.table(f"{BRONZE_CATALOG}.{BRONZE_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n🥈 SILVER LAYER (Cleansed)")
print(f"  Catalog.Schema: {TARGET_CATALOG}.{SILVER_SCHEMA}")
for table in ["medical_claim", "diagnosis", "procedure", "provider", "enrollment"]:
    count = spark.table(f"{TARGET_CATALOG}.{SILVER_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n🥇 GOLD LAYER (Analytics)")
print(f"  Catalog.Schema: {TARGET_CATALOG}.{GOLD_SCHEMA}")
for table in ["patient_summary", "claims_analytics", "cost_analytics"]:
    count = spark.table(f"{TARGET_CATALOG}.{GOLD_SCHEMA}.{table}").count()
    print(f"  {table}: {count:,}")

print("\n" + "="*80)
print("✓ Pipeline completed successfully!")
print("="*80)

In [0]:
%sql
-- Sample queries to explore the Gold layer analytics

-- Top 10 patients by claim count
SELECT 
  patient_id,
  gender,
  age,
  state,
  total_claims,
  enrollment_days,
  first_claim_date,
  last_claim_date
FROM hls_amer_catalog.hcspd_hr_gold_claims.patient_summary
WHERE total_claims > 0
ORDER BY total_claims DESC
LIMIT 10;